# Exploración de BabySlakh (16kHz)

Este notebook tiene como objetivo cargar y visualizar un track de prueba del dataset **BabySlakh**. Vamos a:
1. Cargar y leer los metadatos de un track.
2. Escuchar la mezcla de audio (`mix.wav`).
3. Cargar y visualizar el archivo MIDI original (`all_src.mid`).

In [ ]:
import os
import yaml
import librosa
import librosa.display
import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

# Configuraciones de visualización
plt.rcParams['figure.figsize'] = (14, 5)

### 1. Definir la ruta del track y cargar metadatos

In [ ]:
track_id = "Track00001"
base_dir = "babyslakh_16k"
track_path = os.path.join(base_dir, track_id)

# Cargar metadatos
with open(os.path.join(track_path, "metadata.yaml"), 'r') as f:
    metadata = yaml.safe_load(f)

print(f"--- Metadatos del {track_id} ---")
print(f"Instrumentos en este track:")
for stem_id, stem_info in metadata['stems'].items():
    print(f" - {stem_id}: {stem_info.get('inst_class', 'Unknown')} ({stem_info.get('midi_program_name', 'Unknown')})")

### 2. Escuchar el Audio

In [ ]:
audio_path = os.path.join(track_path, "mix.wav")
y, sr = librosa.load(audio_path, sr=None)
print(f"Audio cargado: {audio_path} | Frecuencia de muestreo (sr): {sr} Hz")

ipd.Audio(y, rate=sr)

### 3. Visualizar el MIDI Original (Ground Truth)

In [ ]:
midi_path = os.path.join(track_path, "all_src.mid")
midi_data = pretty_midi.PrettyMIDI(midi_path)

print(f"Instrumentos en el archivo MIDI:")
for instrument in midi_data.instruments:
    print(f" - {instrument.name} (Program {instrument.program})")

def plot_piano_roll(pm, start_pitch, end_pitch, fs=100):
    # Extraer el piano roll
    piano_roll = pm.get_piano_roll(fs=fs)[start_pitch:end_pitch]
    piano_roll[piano_roll > 0] = 1 # Binarizar para mejor visualización
    
    plt.figure(figsize=(14, 6))
    plt.imshow(piano_roll, aspect='auto', origin='lower', cmap='magma',
               extent=[0, pm.get_end_time(), start_pitch, end_pitch])
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Nota MIDI (Pitch)')
    plt.title('Piano Roll (Ground Truth)')
    plt.colorbar(label='Actividad')
    plt.tight_layout()
    plt.show()

# Graficar notas entre la 24 (C1) y la 108 (C8)
plot_piano_roll(midi_data, start_pitch=24, end_pitch=108)